# Voltage Mode Buck with PPO deep reinforcement learning  

Modified BUCK_VM_PPO example to use GRU network.

## 📝Revisoin History

- **2025-12-29** Initial release.
- **2026-01-08** fixed bug at execution cell. added SAVE_XXX flags. 


## 📁 Directories
```
BUCK_VM_GRU_8x1/
1_train/
├── buck_vm_gru_8x1_gru_gym2.asc                    # LTspice schematics
├── sig_gen.asy                                     # symbol file for signal generator
├── *sig_gen.asc                                     # subcircuit file for signal generator (backup from last LTSpice simulation)
├── *buck_vm_gru_8x1_gru_gym2_param.txt             # Parameter file for LTspice simulation (backup from last LTSpice simulation)
├── *buck_vm_gru_8x1_nn.sp                          # Actor subcircuit file (backup from last LTSpice simulation)
├── *actor_final.pth                                # Actor PyTorch model (backup from last LTSpice simulation)
├── *critic_final.pth                               # Critic PyTorch model (backup from last LTSpice simulation)
└── *gym                                            # Working directly for the training
    ├── *log.csv                                    # Log file of the simulation
    ├── *reward_plot.html                           # Reward plot
    ├── *scatter_plot.html                          # Scatter plot
    ├── *waveform_epXX.html                         # Waveform plot
    ├── *buck_vm_gru_8x1_gru_gym2_param_epXX.txt    # Parameter file for each episode
    ├── *buck_vm_gru_8x1_nn_epXX.sp                 # Actor subcircuit file for each episode
    ├── *actor_epXX.pth                             # Actor model for each episode
    └── *ciritic_epXX.pth                           # Critic model for each episode
*Files/Directory created by this notebook.
```

## ⚔️ LTspice Training Circuit

![buck_vm_gru_8x1_gym2.png](.\buck_vm_gru_8x1_gym2.png)

## 📗Import Libraries

In [ ]:
import os
import shutil
import sys
from pathlib import Path
import numpy as np
import math
import pandas as pd
import torch
from torch import nn, optim
from PyLTSpice import SimRunner, RawRead, LTspice
from pytorch2ltspice import export_model_to_ltspice
from pytorch2ltspice.utils import build_model_from_sequential, sample_on_clock
from pytorch2ltspice.utils.siggen import generate_siggen_asc_asy
from datetime import datetime
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
import ipywidgets as widgets

## ⚙️Configuration


In [ ]:
#Network Configuration
NN_INPUT = 8
NN_OUTPUT = 1
NN_HIDDEN = 16

In [ ]:
# Circuit parameter
VIN_MAX = 250
VIN_MIN = 150
VOUT_MAX = 150
VOUT_MIN = 50
IOUT_MAX = 30
IOUT_MIN = 0.02
LSW = 0.00025    # switching inductance
FSW = 50e3

In [ ]:
# Hyperparameters
STEPS_PER_EPISODE = 2048    
CHUNK_LEN         = 256
EPOCH             = 10
GAMMA             = 0.97
USE_GAE           = True
GAE_LAMBDA        = 0.95
LR_ACTOR          = 3e-4
LR_CRITIC         = 1e-3
CLIP_EPS          = 0.2
ENT_COEF          = 0.005
SIM_TIMEOUT       = 300   #LTSPICE timeout time (sec)

In [ ]:
# Reward function parameters
ALPHA = 10.0            # Weight for squared error term e^2 (steady accuracy)
BETA  = 100.0           # Weight for absolute error term |e| (transient guidance)
K_B_DCM  = 2.0          # Gain for progress penalty in DCM region
W_WORSE  = 10.0*BETA    # Weight for error worsening (wrong direction)
W_NOPROG = 5.0*BETA     # Weight for stagnation (no improvement)
EPS_PROG = 2e-4         # Noise margin for progress detection
DUTYRIPPLE     = 0.1    # Base weight for duty ripple penalty
DUTY_DCM_SCALE = 0.2    # Duty penalty scale in deep DCM (0.1~0.3 recommended)

In [ ]:
# File/Directory
ASCFILE = 'buck_vm_gru_8x1_gym2.asc'
NNFILE = "buck_vm_gru_8x1_nn.sp"
PARAMFILE = 'buck_vm_gru_8x1_gym2_param.txt'
WORKDIR = './gym'
MODEL_ACTOR = 'actor_final.pth'
MODEL_CRITIC = 'critic_final.pth'
SGASCFILE = 'sig_gen.asc'
#create WORKDIR if it doesn't exist
os.makedirs(WORKDIR, exist_ok=True)
#create sig_gen.asy
signals = [0, 0]  #dummy data
generate_siggen_asc_asy(
    signals,
    asc_path=SGASCFILE,
    gen_symbol=True,
)
SAVE_WAVEFORM = False
SAVE_PARAM = False
SAVE_SIGGEN = False
SAVE_SP = False
SAVE_PTH = False

## 🧩Helping Functions

Helping function to create parameter file

In [ ]:
def generate_param_file(params, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for name, value in params.items():
            f.write(f".param {name}={value}\n")
        f.write(f".include {NNFILE}\n")
        f.write("X99 NNin1 NNin2 NNin3 NNin4 NNin5 NNin6 NNin7 NNin8 ctrlclk NNout1 ActorSubckt\n")
        f.write(".save V(ctrlclk) V(NNin1) V(NNin2) V(NNin3) V(NNin4) V(NNin5) V(NNin6) V(NNin7) V(NNin8) V(NNout1) V(NNout1_) V(NNpwm) \n")


Helping functions to create parameter

In [ ]:
def generate_params_random():
    vin_ = np.random.uniform(VIN_MIN, VIN_MAX)
    vref_ = np.random.uniform(VOUT_MIN, VOUT_MAX)
    icrit_= 0.5*(vref_/vin_)*(vin_ - vref_)/(LSW*FSW)
    iload_ = np.random.uniform(IOUT_MIN, IOUT_MAX)
    ro_ = vref_/iload_
    return {
        'vin':  vin_,
        'Lsw':  LSW,
        'ro':   ro_,
        'vref': vref_,
        'fsw':  FSW,
        'VMAX': VIN_MAX,
        'IMAX': IOUT_MAX,        
        'STEPS': STEPS_PER_EPISODE
    }

In [ ]:
def generate_params_DCM():
    vin_ = np.random.uniform(VIN_MIN, VIN_MAX)
    vref_ = np.random.uniform(VOUT_MIN, VOUT_MAX)
    icrit_= 0.5*(vref_/vin_)*(vin_ - vref_)/(LSW*FSW)
    iload_= np.random.uniform(IOUT_MIN,icrit_)
    ro_ = vref_/iload_
    return {
        'vin':  vin_,
        'Lsw':  LSW,
        'ro':   ro_,
        'vref': vref_,
        'fsw':  FSW,
        'VMAX': VIN_MAX,
        'IMAX': IOUT_MAX,        
    'STEPS': STEPS_PER_EPISODE
    }


In [ ]:
def generate_params_CCM():
    vin_ = np.random.uniform(VIN_MIN, VIN_MAX)
    vref_ = np.random.uniform(VOUT_MIN, VOUT_MAX)
    icrit_= 0.5*(vref_/vin_)*(vin_ - vref_)/(LSW*FSW)
    iload_ = np.random.uniform(icrit_, IOUT_MAX)
    ro_ = vref_/iload_
    return {
        'vin':  vin_,
        'Lsw':  LSW,
        'ro':   ro_,
        'vref': vref_,
        'fsw':  FSW,
        'VMAX': VIN_MAX,
        'IMAX': IOUT_MAX,        
        'STEPS': STEPS_PER_EPISODE
    }

In [ ]:
def generate_params_HiDuty():
    vin_ = np.random.uniform(VIN_MIN, VIN_MAX)
    vref_ = np.random.uniform(0.8*vin_, 0.95*vin_)
    icrit_= 0.5*(vref_/vin_)*(vin_ - vref_)/(LSW*FSW)
    iload_ = np.random.uniform(IOUT_MIN, IOUT_MAX)
    ro_ = vref_/iload_
    return {
        'vin':  vin_,
        'Lsw':  LSW,
        'ro':   ro_,
        'vref': vref_,
        'fsw':  FSW,
        'VMAX': VIN_MAX,
        'IMAX': IOUT_MAX,        
        'STEPS': STEPS_PER_EPISODE
    }

## 🧩Create Actor and Critic Networks
Loads .pth file if MODEL_ACTOR and/or MODEL_CRITIC files exists. Otherwise creates new network.

In [ ]:
def build_actor():
    seq = nn.Sequential(
        nn.GRUCell(NN_INPUT, NN_HIDDEN, bias=True),
        nn.Linear(NN_HIDDEN, NN_OUTPUT, bias=True),
        nn.Sigmoid()
    )

    if 'actor_model' in sys.modules:
        del sys.modules['actor_model']

    ActorClass = build_model_from_sequential(
        'Actor',
        seq,
        out_dir=Path('.'),
        out_py_name='actor_model',
        unique_module_name=False,
    )
    return ActorClass()


In [ ]:
def build_critic():
    seq = nn.Sequential(
        nn.Linear(NN_INPUT, 64),
        nn.ReLU(),
        nn.Linear(64, 64),
        nn.ReLU(),
        nn.Linear(64, 1),
    )

    if 'critic_model' in sys.modules:
        del sys.modules['critic_model']

    CriticClass = build_model_from_sequential(
        'Critic',
        seq,
        out_dir=Path('.'),
        out_py_name='critic_model',
        unique_module_name=False,
    )
    return CriticClass()


In [ ]:
# Instantiate networks and optimizers
device = torch.device('cpu')
actor  = build_actor().to(device)
critic = build_critic().to(device)
opt_actor  = optim.Adam(actor.parameters(), lr=LR_ACTOR)
opt_critic = optim.Adam(critic.parameters(), lr=LR_CRITIC)

# Load saved models if available
if os.path.exists(MODEL_ACTOR):
    actor.load_state_dict(torch.load(MODEL_ACTOR, map_location=device))
    print(f"Loaded saved actor model from {MODEL_ACTOR}")
else:
    torch.save(actor.state_dict(), MODEL_ACTOR)
    print(f"Created new actor model and saved to {MODEL_ACTOR}")
if os.path.exists(MODEL_CRITIC):
    critic.load_state_dict(torch.load(MODEL_CRITIC, map_location=device))
    print(f"Loaded saved critic model from {MODEL_CRITIC}")
else:
    torch.save(critic.state_dict(), MODEL_CRITIC)
    print(f"Created new critic model and saved to {MODEL_CRITIC}")

## 🧩LTSpice execution routine

Helping function to extract Status/Action data from .RAW file.
Stops data extraction once duty output gets out of range. 

LTspice execution

In [ ]:

def run_episode(asc_file, work_dir):
    # 1) Create PyLTspice SimRunner instance
    runner = SimRunner(output_folder=work_dir, simulator=LTspice)
    netlist = runner.create_netlist(asc_file)

    # 2) Run simulation
    raw, log = runner.run_now(netlist, timeout=SIM_TIMEOUT)
    raw_data = RawRead(raw)
    df = raw_data.to_dataframe()
    df = sample_on_clock(df, clk='V(ctrlclk)', threshold=0.5, latch_edge='falling')

    # 3) Extract states, actions
    states  = df[[f'V(nnin{i+1})' for i in range(NN_INPUT)]].values[:-1]          # S[t]
    actions = df['V(nnout1_)'].values[:-1]                                        # A[t] (NNOUT1+noise)
    last_state = df[[f'V(nnin{i+1})' for i in range(NN_INPUT)]].values[-1]        # S[STEPS_PER_EPISODE]

    # 4) Reward calculation
    e_t    = df['V(nnin3)'].values[:-1]                 # (Vo-Vref)/Vref[t]
    e_tp1  = df['V(nnin3)'].shift(-1).values[:-1]       # (Vo-Vref)/Vref[t+1]
    dduty  = df['V(nnin5)'].shift(-1).values[:-1]       # duty[t] - duty[t-1]
    vout   = df['V(nnin1)'].values[:-1]                 # Vo/Vmax[t]
    vin    = df['V(nnin6)'].values[:-1]                 # Vin/Vmax[t]
    duty   = df['V(nnin4)'].shift(-1).values[:-1]       # duty[t]
    iout_n = df['V(nnin8)'].values[:-1]                 # Iload/Imax[t]

    # =========================================================
    # (C) DCM level estimation using Icrit
    # =========================================================
    eps = 1e-6
    Ts = 1.0 / FSW
    Vin_V  = vin  * VIN_MAX
    Vout_V = vout * VIN_MAX
    Iout_A = iout_n * IOUT_MAX

    dIL = (np.maximum(0.0, Vin_V - Vout_V) / (LSW + eps)) * np.clip(duty, 0.0, 1.0) * Ts
    Icrit = 0.5 * dIL
    dcm_level = np.clip((Icrit - Iout_A) / (Icrit + eps), 0.0, 1.0)   # 0:CCM, 1:deep DCM

    # =========================================================
    # (A) Base error reward
    # =========================================================
    r_err_abs = -BETA * np.abs(e_tp1)
    r_err_sq  = -ALPHA * (e_tp1 ** 2)

    # =========================================================
    # (B) Progress reward using Δe
    # =========================================================
    deltae = e_tp1 - e_t
    worsening = np.maximum(0.0, np.sign(e_t) * deltae)   # error getting worse
    no_prog   = np.maximum(0.0, np.abs(e_tp1) - np.abs(e_t) + EPS_PROG)
    wB = (1.0 + K_B_DCM * dcm_level)
    r_worse  = -(W_WORSE  * wB) * worsening
    r_noprog = -(W_NOPROG * wB) * no_prog

    # =========================================================
    # Δduty penalty with DCM gating
    # =========================================================
    w_duty = (1.0 - dcm_level) + DUTY_DCM_SCALE * dcm_level
    r_duty = -(DUTYRIPPLE * w_duty) * (dduty ** 2)

    # =========================================================
    # Total reward
    # =========================================================
    rewards = r_err_abs + r_err_sq + r_worse + r_noprog + r_duty

    # =========================================================
    # Reward breakdown log (episode-level statistics)
    # =========================================================
    reward_log = {
        "r_err_abs": np.mean(r_err_abs),
        "r_err_sq":  np.mean(r_err_sq),
        "r_worse":   np.mean(r_worse),
        "r_noprog":  np.mean(r_noprog),
        "r_duty":    np.mean(r_duty),
        "r_total":   np.mean(rewards),

        # diagnostics
        "dcm_level": np.mean(dcm_level),
        "worsening": np.mean(worsening),
        "no_prog":   np.mean(no_prog),
        "dduty":     np.mean(np.abs(dduty)),
    }

    # 5) Crean PyLTspice files
    runner.cleanup_files()

    return states, actions, rewards, last_state, df, reward_log


## 🧩PPO Update Step

In [ ]:
def ppo_update(states, actions, rewards, noise_std, last_state, chunk_len=64, epochs=10):
    # 0) Convert to tensors
    states_t  = torch.tensor(states,  dtype=torch.float32, device=device)          # (T, D)
    actions_t = torch.tensor(actions, dtype=torch.float32, device=device).unsqueeze(-1)  # (T, 1)
    rewards_t = torch.tensor(rewards, dtype=torch.float32, device=device)          # (T,)
    last_state_t = torch.tensor(last_state, dtype=torch.float32, device=device)    # (D,) or (1,D)
    if last_state_t.dim() == 1:
        last_state_t = last_state_t.unsqueeze(0)                                   # (1, D)

    dataset_size = states_t.size(0)

    # 1) old policy log-prob + actor init states (for RNN TBPTT)
    actor_init_states = []
    with torch.no_grad():
        actor_state = None
        mu_steps = []
        for t in range(dataset_size):
            actor_init_states.append(actor.clone_state(actor_state))               # state BEFORE s[t]
            step_input = states_t[t].unsqueeze(0)                                  # (1, D)
            mu_step, actor_state = actor.step(step_input, actor_state)             # (1, 1)
            mu_steps.append(mu_step)

        mu = torch.cat(mu_steps, dim=0)                                            # (T, 1)
        actions_py = mu.detach().cpu().numpy().squeeze()
        dist_old = torch.distributions.Normal(mu, noise_std)
        old_log_probs = dist_old.log_prob(actions_t).squeeze(-1)                   # (T,)

        # 2) values (MLP critic, no hidden)
        values = critic(states_t).squeeze(-1)                                      # (T,)
        last_value = critic(last_state_t).squeeze(-1).squeeze(0)                   # scalar

        # 3) returns & advantages
        if USE_GAE:
            next_values = torch.empty_like(values)
            next_values[:-1] = values[1:]
            next_values[-1]  = last_value
            deltas = rewards_t + GAMMA * next_values - values

            advantages = torch.zeros_like(rewards_t)
            gae = 0.0
            for t in reversed(range(dataset_size)):
                gae = deltas[t] + GAMMA * GAE_LAMBDA * gae
                advantages[t] = gae
            returns = advantages + values
        else:
            returns = torch.zeros_like(rewards_t)
            G = 0.0
            for t in reversed(range(dataset_size)):
                G = rewards_t[t] + GAMMA * G
                returns[t] = G
            advantages = returns - values

        # Normalize advantages (detach OK)
        advantages = (advantages - advantages.mean()) / advantages.std().clamp_min(1e-8)
        advantages = advantages.detach()
        returns = returns.detach()

    # 4) PPO optimize
    for epoch in range(epochs):
        for start in range(0, dataset_size, chunk_len):
            end = min(start + chunk_len, dataset_size)

            mb_states        = states_t[start:end]             # (L, D)  ※L=chunk_len
            mb_actions       = actions_t[start:end]            # (L, 1)
            mb_old_log_probs = old_log_probs[start:end]        # (L,)
            mb_returns       = returns[start:end]              # (L,)
            mb_advantages    = advantages[start:end]           # (L,)

            init_actor_state = actor.clone_state(actor_init_states[start])

            # Critic loss (MLP)
            mb_values = critic(mb_states).squeeze(-1)          # (L,)
            loss_critic = nn.MSELoss()(mb_values, mb_returns)

            # Actor loss (PPO-clip)
            mu_new = actor(mb_states, h=init_actor_state)      # (L, 1)
            dist_new = torch.distributions.Normal(mu_new, noise_std)
            log_probs_new = dist_new.log_prob(mb_actions).squeeze(-1)  # (L,)
            ratios = torch.exp(log_probs_new - mb_old_log_probs)
            surr1 = ratios * mb_advantages
            surr2 = torch.clamp(ratios, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS) * mb_advantages
            entropy = dist_new.entropy().squeeze(-1)
            loss_actor = -torch.min(surr1, surr2).mean() - ENT_COEF * entropy.mean()

            opt_actor.zero_grad()
            loss_actor.backward()
            torch.nn.utils.clip_grad_norm_(actor.parameters(), max_norm=0.5)
            opt_actor.step()

            opt_critic.zero_grad()
            loss_critic.backward()
            torch.nn.utils.clip_grad_norm_(critic.parameters(), max_norm=0.5)
            opt_critic.step()

    return (
        dataset_size,
        loss_actor.item(),
        loss_critic.item(),
        actions_py,
        values.detach().cpu().numpy(),
        returns.detach().cpu().numpy(),
        advantages.detach().cpu().numpy()
    )


## 📉Training Status Plot

In [ ]:
fig_reward = go.Figure()
fig_reward.add_trace(go.Scatter(x=[], y=[], mode='lines+markers', name='Average Reward', yaxis='y1'))
fig_reward.add_trace(go.Scatter(x=[], y=[], mode='lines+markers', name='Noise Std', yaxis='y2'))
fig_reward.update_layout(xaxis=dict(title='Episode'), yaxis=dict(title='Average Reward'), yaxis2=dict(title='Noise Std', overlaying='y', side='right'), legend=dict(x=0, y=1.2, orientation='h'))

fig_scatter = go.Figure()
fig_scatter.add_trace(go.Scatter(x=[], y=[], mode='markers', marker=dict(size=10, color=[], colorscale='Viridis', colorbar=dict(title='Reward'), showscale=True), text=[], hoverinfo='text'))
fig_scatter.update_layout(title='Vout/Vin vs Iout/Icrit (Colored by Average Reward)', xaxis_title='Iout/Icrit', yaxis_title='Vout/Vin')

fig_nn = go.Figure()
fig_nn.add_trace(go.Scatter(x=[], y=[], name='nnin1(Vo/Vmax)', yaxis='y1', mode='lines+markers'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='nnin7(Vref/Vmax)', yaxis='y1', mode='lines'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='nnpwm', yaxis='y1', mode='lines+markers'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='nnout1', yaxis='y1', mode='lines+markers'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='nnout1(python)', yaxis='y1', mode='lines+markers'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='rewards', yaxis='y2', mode='lines+markers'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='values', yaxis='y2', mode='lines+markers'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='returns', yaxis='y2', mode='lines+markers'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='advantages', yaxis='y2', mode='lines+markers'))
fig_nn.update_layout(xaxis=dict(title='Step'), yaxis=dict(title='NNIO'), yaxis2=dict(title='Rewards', overlaying='y', side='right'))
fig_nn.update_layout(legend=dict(orientation='h', x=0.5, y=-0.3, xanchor='center', yanchor='top'), height=500)
t = list(range(STEPS_PER_EPISODE))
for trace in fig_nn.data:
    trace.x = t

fig_nn2 = go.Figure()
fig_nn2.add_trace(go.Scatter(x=[], y=[], name='nnin1(Vo/Vmax)', yaxis='y1', mode='lines+markers'))
fig_nn2.add_trace(go.Scatter(x=[], y=[], name='nnin7(Vref/Vmax)', yaxis='y1', mode='lines'))
fig_nn2.add_trace(go.Scatter(x=[], y=[], name='nnpwm', yaxis='y1', mode='lines+markers'))
fig_nn2.add_trace(go.Scatter(x=[], y=[], name='nnout1', yaxis='y1', mode='lines+markers'))
fig_nn2.add_trace(go.Scatter(x=[], y=[], name='rewards', yaxis='y2', mode='lines+markers'))
fig_nn2.update_layout(xaxis=dict(title='Step'), yaxis=dict(title='NNIO'), yaxis2=dict(title='Rewards', overlaying='y', side='right'), legend=dict(x=0, y=1.2, orientation='h'))
t = list(range(STEPS_PER_EPISODE))
for trace in fig_nn2.data:
    trace.x = t


## ♻️Training Loop

In [ ]:
#reload or newly create log.csv
if os.path.exists(WORKDIR+"/log.csv"):
    summary_df = pd.read_csv(WORKDIR+"/log.csv") 
    ep_idx = len(summary_df) + 1

else:
    summary_df = pd.DataFrame(columns=[
        'episode','sim time','loss_actor','loss_critic',
        'vin','Lsw','ro','vref','fsw','std','steps','chunk','epoch',
        'Vout/Vin','Iout/Icrit',
        # reward breakdown
        'r_err_abs','r_err_sq','r_worse','r_noprog','r_duty','r_total',
        # diagnostics
        'dcm_level','worsening','no_prog','dduty'
    ])
    ep_idx = 1

def train_loop(base_ep, num_ep, noise_stds, param_fn):        
    # Remove episode data from summary_df starting at base_ep onwards
    global summary_df
    summary_df = summary_df[summary_df['episode'] < base_ep].reset_index(drop=True)

    ep_cnt = 0
    # Main loop
    for ep in range(base_ep , base_ep + num_ep):
        try:
            sim_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            print(f">>> {sim_time}")

            # 1) Save actor/critic files
            if SAVE_PTH == True:
                torch.save(actor.state_dict(), WORKDIR+f"/actor_ep{ep}.pth")
                torch.save(critic.state_dict(), WORKDIR+f"/critic_ep{ep}.pth")
            torch.save(actor.state_dict(), MODEL_ACTOR)
            torch.save(critic.state_dict(), MODEL_CRITIC)


            # 2) Export current actor to SPICE subckt
            export_model_to_ltspice(actor.model, filename=NNFILE, subckt_name='ActorSubckt', verbose=False)
            shutil.copy2(NNFILE, WORKDIR)
            if SAVE_SP == True:
                name, ext = os.path.splitext(NNFILE)   
                shutil.copy2(NNFILE, f"{WORKDIR}/{name}_ep{ep}{ext}")   

            # 3) Generate a new noise sequence and create SigGen subckt
            idx_noise = ep - base_ep
            if idx_noise >= len(noise_stds):
                noise_std = noise_stds[-1]
            else:
                noise_std = noise_stds[idx_noise]
            noises = np.random.normal(loc=0.0, scale=noise_std, size=STEPS_PER_EPISODE)
            generate_siggen_asc_asy(
                signals=noises,
                asc_path=SGASCFILE,
                gen_symbol=False,
                subckt_name='sig_gen',
            )
            shutil.copy2(SGASCFILE, WORKDIR)
            if SAVE_SIGGEN == True:
                name, ext = os.path.splitext(SGASCFILE)   
                shutil.copy2(SGASCFILE, f"{WORKDIR}/{name}_ep{ep}{ext}")   
            shutil.copy2(os.path.splitext(SGASCFILE)[0] + ".asy", WORKDIR)
             
            # 4) Generate Parameter file
            params = param_fn()
            generate_param_file(params, PARAMFILE)
            shutil.copy2(PARAMFILE, WORKDIR)
            if SAVE_PARAM == True:
                name, ext = os.path.splitext(PARAMFILE)   
                shutil.copy2(PARAMFILE, f"{WORKDIR}/{name}_ep{ep}{ext}")   
            
            # 5) Run one full episode in LTspice
            vout_vin = params['vref'] / params['vin']
            iout_icrit = params['vref'] / params['ro'] / (0.5 * params['vref'] * (params['vin'] - params['vref']) / params['fsw'] / params['Lsw'] / params['vin'])
            print(f"[Ep{ep}/{base_ep+num_ep-1}] Vout/Vin:{vout_vin:.2f}, Iout/Icrit:{iout_icrit:.3f}, std:{noise_std:.4f}")
            states, actions, rewards, last_state, df, reward_log = run_episode(ASCFILE, WORKDIR)

            # 6) Perform PPO update with collected data
            dataset_size, loss_actor, loss_critic, actions_py, values, returns, advantages = ppo_update(states, actions, rewards, noise_std, last_state, chunk_len=CHUNK_LEN, epochs=EPOCH)

            # 7) Compute and print summary statistics
            print(f"[Ep{ep}/{base_ep+num_ep-1}]",
                f"Loss_actor: {loss_actor:.3f}, "
                f"Loss_critic: {loss_critic:.3f}")

            # 8) Update&Save episode graph
            fig_nn.data[0].y = df['V(nnin1)']
            fig_nn.data[1].y = df['V(nnin7)']
            fig_nn.data[2].y = df['V(nnpwm)']
            fig_nn.data[3].y = df['V(nnout1)']
            fig_nn.data[4].y = actions_py
            fig_nn.data[5].y = rewards
            fig_nn.data[6].y = values
            fig_nn.data[7].y = returns
            fig_nn.data[8].y = advantages
            fig_nn.update_layout(title_text=f"Ep{ep}: Vout/Vin={vout_vin:.2f}, Iout/Icrit={iout_icrit:.2f}, std={noise_std:.4f}")
            fig_nn2.data[0].y = df['V(nnin1)']
            fig_nn2.data[1].y = df['V(nnin7)']
            fig_nn2.data[2].y = df['V(nnpwm)']
            fig_nn2.data[3].y = df['V(nnout1)']
            fig_nn2.data[4].y = rewards
            fig_nn2.update_layout(title_text=f"Ep{ep}: Vout/Vin={vout_vin:.2f}, Iout/Icrit={iout_icrit:.2f}, std={noise_std:.4f}")
            if SAVE_WAVEFORM == True:
                html_path = WORKDIR + f"/waveform_ep{ep}.html"
                fig_nn2.write_html(html_path, include_plotlyjs='cdn')

            # 9) Append summary
            summary_df.loc[len(summary_df)] = {
                'episode':      ep,
                'sim time':     sim_time,
                'loss_actor':   loss_actor,
                'loss_critic':  loss_critic,
                'vin':          params['vin'],
                'Lsw':          params['Lsw'],
                'ro':           params['ro'],
                'vref':         params['vref'],
                'fsw':          params['fsw'],
                'std':          noise_std,
                'steps':        dataset_size,
                'chunk':        CHUNK_LEN,
                'epoch':        EPOCH,
                'Vout/Vin':     vout_vin,
                'Iout/Icrit':   iout_icrit,
                **reward_log
            }

            # 10) Update/Save learning curve plot
            fig_reward.data[0].x = summary_df['episode']
            fig_reward.data[0].y = summary_df['r_total']
            fig_reward.data[1].x = summary_df['episode']
            fig_reward.data[1].y = summary_df['std']
            reward_html_path = os.path.join(WORKDIR, "reward_plot.html")
            fig_reward.write_html(reward_html_path, include_plotlyjs='cdn')

            # 11) Update/Save Scatter plot
            fig_scatter.data[0].x = summary_df['Iout/Icrit']
            fig_scatter.data[0].y = summary_df['Vout/Vin']
            fig_scatter.data[0].marker.color = summary_df['r_total']
            fig_scatter.data[0].text = summary_df['episode']
            scatter_html_path = os.path.join(WORKDIR, "scatter_plot.html")
            fig_scatter.write_html(scatter_html_path, include_plotlyjs='cdn')

            # 12) Update Display
            with out:
                out.clear_output(wait=True)
                display(fig_reward)
                display(fig_scatter)
                display(fig_nn)

            # 13) Save summary to CSV
            episode_csv = os.path.join(WORKDIR, 'log.csv')
            summary_df.to_csv(episode_csv, index=False)    

            # 14) Save actor/critic files
            torch.save(actor.state_dict(), MODEL_ACTOR)
            torch.save(critic.state_dict(), MODEL_CRITIC)

            ep_cnt += 1

        except Exception as e:
            print(f"[Ep{ep}/{base_ep+num_ep-1}] ERROR: Failed with exception: {e}")
            ep_cnt += 1
            continue

    return ep_cnt


##  🧠Training Display

In [ ]:
# Render via MIME; avoid FigureWidget and avoid per-ep new outputs
pio.renderers.default = "plotly_mimetype"

# Single output area for all charts
out = widgets.Output(layout={"border":"1px solid #ddd"})

with out:
    out.clear_output(wait=True)
    display(fig_reward)
    display(fig_scatter)
    display(fig_nn)

out

##  🧠Training Steps(Example)

In [ ]:
NUM_EPISODES = 100

#noise
MAX_STD = 0.1
MIN_STD = 0.05
noise_stds = [MIN_STD + 0.5*(MAX_STD - MIN_STD)*(1 + math.cos(math.pi * ep/(NUM_EPISODES-1))) for ep in range(NUM_EPISODES)]

#Reward parameter
ALPHA = 10.0            # Weight for squared error term e^2 (steady accuracy)
BETA  = 100.0           # Weight for absolute error term |e| (transient guidance)
K_B_DCM  = 2.0          # Gain for progress penalty in DCM region
W_WORSE  = 10.0*BETA    # Weight for error worsening (wrong direction)
W_NOPROG = 5.0*BETA     # Weight for stagnation (no improvement)
EPS_PROG = 2e-4         # Noise margin for progress detection
DUTYRIPPLE     = 0.1    # Base weight for duty ripple penalty
DUTY_DCM_SCALE = 0.2    # Duty penalty scale in deep DCM (0.1~0.3 recommended)

#excute training
for i in range(NUM_EPISODES//10):
    ep_idx += train_loop(ep_idx, 5, noise_stds[i*10:], generate_params_random)
    ep_idx += train_loop(ep_idx, 5, noise_stds[(i*10+5):], generate_params_DCM)

In [ ]:
NUM_EPISODES = 200

#noise
MAX_STD = 0.05
MIN_STD = 0.04
noise_stds = [MIN_STD + 0.5*(MAX_STD - MIN_STD)*(1 + math.cos(math.pi * ep/(NUM_EPISODES-1))) for ep in range(NUM_EPISODES)]

#Reward parameter
ALPHA = 10.0            # Weight for squared error term e^2 (steady accuracy)
BETA  = 100.0           # Weight for absolute error term |e| (transient guidance)
K_B_DCM  = 2.0          # Gain for progress penalty in DCM region
W_WORSE  = 10.0*BETA    # Weight for error worsening (wrong direction)
W_NOPROG = 5.0*BETA     # Weight for stagnation (no improvement)
EPS_PROG = 2e-4         # Noise margin for progress detection
DUTYRIPPLE     = 0.1    # Base weight for duty ripple penalty
DUTY_DCM_SCALE = 0.2    # Duty penalty scale in deep DCM (0.1~0.3 recommended)

#excute training
for i in range(NUM_EPISODES//10):
    ep_idx += train_loop(ep_idx, 2, noise_stds[i*10:], generate_params_random)
    ep_idx += train_loop(ep_idx, 8, noise_stds[(i*10+2):], generate_params_DCM)

In [ ]:
NUM_EPISODES = 200

#noise
MAX_STD = 0.04
MIN_STD = 0.01
noise_stds = [MIN_STD + 0.5*(MAX_STD - MIN_STD)*(1 + math.cos(math.pi * ep/(NUM_EPISODES-1))) for ep in range(NUM_EPISODES)]

#Reward parameter
ALPHA = 10.0            # Weight for squared error term e^2 (steady accuracy)
BETA  = 100.0           # Weight for absolute error term |e| (transient guidance)
K_B_DCM  = 2.0          # Gain for progress penalty in DCM region
W_WORSE  = 10.0*BETA    # Weight for error worsening (wrong direction)
W_NOPROG = 5.0*BETA     # Weight for stagnation (no improvement)
EPS_PROG = 2e-4         # Noise margin for progress detection
DUTYRIPPLE     = 0.1    # Base weight for duty ripple penalty
DUTY_DCM_SCALE = 0.2    # Duty penalty scale in deep DCM (0.1~0.3 recommended)

#excute training
for i in range(NUM_EPISODES//10):
    ep_idx += train_loop(ep_idx, 2, noise_stds[i*10:], generate_params_random)
    ep_idx += train_loop(ep_idx, 8, noise_stds[(i*10+2):], generate_params_DCM)